# RamanBench — Contributing a New Dataset

This notebook walks through the process of contributing a new Raman spectroscopy
dataset to the RamanBench ecosystem.

## Ecosystem

| Resource                         | Link                                                                                               |
|----------------------------------|----------------------------------------------------------------------------------------------------|
| **raman-data** (dataset package) | [GitHub](https://github.com/ml-lab-htw/raman_data) · [PyPI](https://pypi.org/project/raman-data/)  |
| **raman-bench** (benchmark)      | [GitHub](https://github.com/ml-lab-htw/RamanBench) · [PyPI](https://pypi.org/project/raman-bench/) |
| **Live Leaderboard**             | [HuggingFace](https://huggingface.co/spaces/ml-lab-htw/RamanBench)                                 |
| **Paper**                        | [arXiv TBD](https://arxiv.org/abs/TBD)                                                             |

## Overview

The contribution process has three steps:

1. **Prepare your data** — upload to HuggingFace Datasets or Zenodo (CC BY 4.0)
2. **Add a loader** — open a PR in [raman-data](https://github.com/ml-lab-htw/raman_data)
3. **Request inclusion** — open an issue in [RamanBench](https://github.com/ml-lab-htw/RamanBench/issues/new?template=dataset_submission.md)

## Step 1: Prepare your data

### Required format

Your dataset should be structured as a Pandas DataFrame:
- **Rows** = samples (one spectrum per row)
- **Columns** = wavenumber values (floats, in cm⁻¹)
- **Last column(s)** = target values (regression) or labels (classification)

### Inclusion criteria

| Criterion | Details |
|---|---|
| **Freely accessible** | Publicly available (HuggingFace, Zenodo, Kaggle, etc.) under an open license (CC BY 4.0 or more permissive) |
| **Experimentally acquired** | Real instrument measurements — no simulated or synthetic spectra |
| **Supervised labels** | At least one regression target or classification label per spectrum |
| **Minimum size** | ≥ 10 labeled spectra total; for classification ≥ 9 spectra per class (rare classes removed; excluded if < 2 classes remain) |
| **Learnability** | Regression: R² > 0.05 with at least one model; Classification: ΔF1 > 0.05 above majority-class baseline (checked automatically during integration) |
| **Citation** | Published paper or preprint with DOI |


In [1]:
import numpy as np
import pandas as pd

# Example: create a dataset from scratch
rng = np.random.default_rng(42)
n_samples, n_wavenumbers = 100, 500
wavenumbers = np.linspace(400, 2000, n_wavenumbers)

# Spectra as rows, wavenumbers as column names
spectra_df = pd.DataFrame(
    rng.standard_normal((n_samples, n_wavenumbers)),
    columns=wavenumbers.astype(str),
)
# Add target column(s)
spectra_df['concentration_mM'] = rng.uniform(0, 100, n_samples)

print(spectra_df.shape)
spectra_df.head()

(100, 501)


,400.0,403.2064128256513,406.4128256513026,409.6192384769539,412.8256513026052,416.03206412825654,419.2384769539078,422.4448897795591,425.65130260521045,428.85771543086173,...,1974.3486973947897,1977.5551102204408,1980.7615230460922,1983.9679358717435,1987.1743486973949,1990.3807615230462,1993.5871743486975,1996.7935871743487,2000.0,concentration_mM
0,0.304717,-1.039984,0.750451,0.940565,-1.951035,-1.302180,0.127840,-0.316243,-0.016801,-0.853044,...,0.366531,-0.286249,0.453966,-0.308673,0.935547,-1.831406,-0.335607,-1.990812,-1.495061,36.967474
1,1.363862,0.895185,-0.719480,-1.502503,-2.964529,-0.543496,2.420415,0.434884,-0.559572,0.465080,...,1.479275,1.794370,1.314808,-0.109734,0.352720,0.766823,0.121178,0.130764,0.823753,74.931852
2,-0.059283,-0.729287,-0.414473,0.633910,0.002993,0.340210,0.670079,-0.374841,0.756248,0.378843,...,-0.022685,-0.622540,-1.071701,-0.353615,-1.103989,0.322797,-1.150085,0.711977,-1.181479,63.854453
3,-0.566295,-0.624603,1.325147,0.330290,-0.211734,0.498719,-2.107192,-0.043134,1.997514,0.132347,...,0.545323,0.231660,0.965170,0.087868,0.392011,0.303837,-0.091540,-0.530822,2.214153,85.483957
4,-0.451951,-0.665878,0.434010,0.251854,-1.404792,1.122678,-0.094195,-1.110931,1.188784,0.625676,...,-0.701767,0.765955,-0.761992,1.090660,2.210641,-0.122084,-2.192543,-0.548073,1.231875,11.100893


## Step 2: Upload to HuggingFace Datasets

```python
from datasets import Dataset

hf_dataset = Dataset.from_pandas(spectra_df)
hf_dataset.push_to_hub('your-username/my_raman_dataset')
```

Or upload to Zenodo as a CSV/Parquet file.

## Step 3: Add a loader to raman-data

Fork [ml-lab-htw/raman_data](https://github.com/ml-lab-htw/raman_data) and add an entry:

In `raman_data/loaders/HuggingFaceLoader.py`:
```python
DATASETS = {
    # ... existing datasets ...
    'my_compound_concentration': DatasetInfo(
        name='My Compound Raman Dataset',
        task_type=TASK_TYPE.Regression,
        application_type=APPLICATION_TYPE.Chemical,
        source='your-username/my_raman_dataset',
        license='CC BY 4.0',
        citation='Author et al. (2026). doi:10.xxxx/xxxx',
    ),
}
```

## Step 4: Verify the loader works

```python
from raman_data import raman_data

dataset = raman_data('my_compound_concentration')
print(dataset.spectra.shape)
print(dataset.target_names)
```

## Step 5: Request inclusion in RamanBench

Open an issue using the [dataset submission template](https://github.com/ml-lab-htw/RamanBench/issues/new?template=dataset_submission.md).

Once your raman-data PR is merged and a new release is published, we'll add
your dataset to `configs/datasets/` and evaluate all baseline models on it.

## Existing new datasets

For inspiration, see [NEW_DATASETS.md](../NEW_DATASETS.md) which documents
17 datasets released alongside RamanBench v0.1, including fermentation monitoring,
metabolite analysis, and gasoline characterisation datasets.